# 📊 Exploratory Data Analysis (EDA)
## Lung Cancer Gene Expression Data

This notebook explores the TCGA lung cancer dataset containing:
- **LUAD**: Lung Adenocarcinoma (576 samples)
- **LUSC**: Lung Squamous Cell Carcinoma (553 samples)

---

## 1️⃣ Setup & Load Data

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Plot settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("Libraries loaded successfully!")

In [ ]:
# Load the combined dataset
df = pd.read_csv('../data/raw/lung_cancer_combined.csv', index_col='sample_id')

print(f"Dataset Shape: {df.shape}")
print(f"  - Samples: {df.shape[0]:,}")
print(f"  - Features (genes + label): {df.shape[1]:,}")

## 2️⃣ Basic Data Inspection

In [ ]:
# View first few rows
df.head()

In [ ]:
# Data types
print("Data Types:")
print(f"  - cancer_type: {df['cancer_type'].dtype}")
print(f"  - Gene columns: {df.iloc[:, 1:5].dtypes.unique()}")

In [ ]:
# Check for missing values
missing = df.isnull().sum().sum()
missing_pct = (df.isnull().sum().sum() / df.size) * 100

print(f"Missing Values: {missing:,} ({missing_pct:.2f}%)")

# Columns with most missing values
missing_per_col = df.isnull().sum()
cols_with_missing = missing_per_col[missing_per_col > 0].sort_values(ascending=False)
if len(cols_with_missing) > 0:
    print(f"\nTop 10 columns with missing values:")
    print(cols_with_missing.head(10))
else:
    print("No missing values found! ✅")

## 3️⃣ Class Distribution

In [ ]:
# Class distribution
class_counts = df['cancer_type'].value_counts()
print("Class Distribution:")
print(class_counts)
print(f"\nClass Balance Ratio: {class_counts.min() / class_counts.max():.2f}")

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
colors = ['#3498db', '#e74c3c']
axes[0].bar(class_counts.index, class_counts.values, color=colors, edgecolor='black')
axes[0].set_xlabel('Cancer Type')
axes[0].set_ylabel('Number of Samples')
axes[0].set_title('Sample Distribution by Cancer Type')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%', 
            colors=colors, explode=(0.02, 0.02), shadow=True, startangle=90)
axes[1].set_title('Class Proportion')

plt.tight_layout()
plt.show()

## 4️⃣ Gene Expression Statistics

In [ ]:
# Separate features and labels
X = df.drop('cancer_type', axis=1)
y = df['cancer_type']

print(f"Feature matrix shape: {X.shape}")
print(f"Label vector shape: {y.shape}")

In [ ]:
# Gene expression statistics
gene_stats = X.describe().T
gene_stats.columns = ['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']

print("Gene Expression Statistics (across all genes):")
print(f"  Mean expression range: {gene_stats['mean'].min():.2f} to {gene_stats['mean'].max():.2f}")
print(f"  Std deviation range: {gene_stats['std'].min():.2f} to {gene_stats['std'].max():.2f}")
print(f"  Overall min: {gene_stats['min'].min():.2f}")
print(f"  Overall max: {gene_stats['max'].max():.2f}")

In [ ]:
# Distribution of mean gene expression
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mean expression distribution
axes[0].hist(gene_stats['mean'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(gene_stats['mean'].median(), color='red', linestyle='--', label=f'Median: {gene_stats["mean"].median():.2f}')
axes[0].set_xlabel('Mean Expression (log2)')
axes[0].set_ylabel('Number of Genes')
axes[0].set_title('Distribution of Mean Gene Expression')
axes[0].legend()

# Variance distribution
gene_variance = X.var()
axes[1].hist(gene_variance, bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[1].axvline(gene_variance.median(), color='red', linestyle='--', label=f'Median: {gene_variance.median():.2f}')
axes[1].set_xlabel('Variance')
axes[1].set_ylabel('Number of Genes')
axes[1].set_title('Distribution of Gene Expression Variance')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5️⃣ Sample-Level Analysis

In [ ]:
# Expression distribution per sample
sample_means = X.mean(axis=1)
sample_stds = X.std(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mean expression by cancer type
for cancer_type, color in zip(['LUAD', 'LUSC'], colors):
    mask = y == cancer_type
    axes[0].hist(sample_means[mask], bins=30, alpha=0.6, label=cancer_type, color=color, edgecolor='black')
axes[0].set_xlabel('Mean Expression per Sample')
axes[0].set_ylabel('Number of Samples')
axes[0].set_title('Distribution of Mean Expression by Cancer Type')
axes[0].legend()

# Std by cancer type
for cancer_type, color in zip(['LUAD', 'LUSC'], colors):
    mask = y == cancer_type
    axes[1].hist(sample_stds[mask], bins=30, alpha=0.6, label=cancer_type, color=color, edgecolor='black')
axes[1].set_xlabel('Std Deviation per Sample')
axes[1].set_ylabel('Number of Samples')
axes[1].set_title('Distribution of Expression Std by Cancer Type')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6️⃣ PCA Visualization

Let's see if the two cancer types are separable in lower dimensions.

In [ ]:
# Standardize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply PCA
pca = PCA(n_components=50)  # Get first 50 components
X_pca = pca.fit_transform(X_scaled)

print(f"Explained variance by first 10 components:")
for i in range(10):
    print(f"  PC{i+1}: {pca.explained_variance_ratio_[i]*100:.2f}%")
print(f"\nCumulative variance (first 50 components): {sum(pca.explained_variance_ratio_)*100:.2f}%")

In [ ]:
# Visualize explained variance
plt.figure(figsize=(10, 5))
cumulative_var = np.cumsum(pca.explained_variance_ratio_) * 100

plt.bar(range(1, 51), pca.explained_variance_ratio_ * 100, alpha=0.7, label='Individual')
plt.plot(range(1, 51), cumulative_var, 'r-o', markersize=4, label='Cumulative')
plt.axhline(y=90, color='green', linestyle='--', alpha=0.7, label='90% threshold')

# Find number of components for 90% variance
n_90 = np.argmax(cumulative_var >= 90) + 1
plt.axvline(x=n_90, color='purple', linestyle='--', alpha=0.7)
plt.text(n_90 + 1, 50, f'{n_90} PCs for 90%', fontsize=10)

plt.xlabel('Principal Component')
plt.ylabel('Explained Variance (%)')
plt.title('PCA Explained Variance')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 2D PCA scatter plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# PC1 vs PC2
for cancer_type, color in zip(['LUAD', 'LUSC'], colors):
    mask = y == cancer_type
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, label=cancer_type, alpha=0.6, s=30)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].set_title('PCA: PC1 vs PC2')
axes[0].legend()

# PC1 vs PC3
for cancer_type, color in zip(['LUAD', 'LUSC'], colors):
    mask = y == cancer_type
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 2], c=color, label=cancer_type, alpha=0.6, s=30)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.1f}%)')
axes[1].set_title('PCA: PC1 vs PC3')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7️⃣ Top Variable Genes

Find genes with highest variance (most informative for classification).

In [ ]:
# Get top 20 most variable genes
gene_variance = X.var().sort_values(ascending=False)
top_genes = gene_variance.head(20)

print("Top 20 Most Variable Genes:")
for i, (gene, var) in enumerate(top_genes.items(), 1):
    print(f"  {i:2d}. {gene}: variance = {var:.2f}")

In [ ]:
# Visualize top variable genes
plt.figure(figsize=(12, 6))
plt.barh(range(len(top_genes)), top_genes.values, color='steelblue', edgecolor='black')
plt.yticks(range(len(top_genes)), top_genes.index)
plt.xlabel('Variance')
plt.ylabel('Gene')
plt.title('Top 20 Most Variable Genes')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 8️⃣ Differential Expression Preview

Quick look at genes that differ between LUAD and LUSC.

In [ ]:
# Calculate mean expression for each cancer type
luad_mean = X[y == 'LUAD'].mean()
lusc_mean = X[y == 'LUSC'].mean()

# Calculate fold change (difference in log2 space = log2 fold change)
fold_change = luad_mean - lusc_mean

# Get top differentially expressed genes
top_up_in_luad = fold_change.sort_values(ascending=False).head(10)
top_up_in_lusc = fold_change.sort_values(ascending=True).head(10)

print("Top 10 Genes HIGHER in LUAD:")
for gene, fc in top_up_in_luad.items():
    print(f"  {gene}: log2FC = {fc:.2f}")

print("\nTop 10 Genes HIGHER in LUSC:")
for gene, fc in top_up_in_lusc.items():
    print(f"  {gene}: log2FC = {fc:.2f}")

In [ ]:
# Visualize top differential genes as heatmap
top_diff_genes = list(top_up_in_luad.index) + list(top_up_in_lusc.index)

# Sample 50 random samples from each class for visualization
np.random.seed(42)
luad_samples = np.random.choice(df[y == 'LUAD'].index, size=50, replace=False)
lusc_samples = np.random.choice(df[y == 'LUSC'].index, size=50, replace=False)
selected_samples = list(luad_samples) + list(lusc_samples)

# Create heatmap data
heatmap_data = X.loc[selected_samples, top_diff_genes]

# Create row colors
row_colors = pd.Series(['#3498db'] * 50 + ['#e74c3c'] * 50, index=selected_samples, name='Type')

# Plot
plt.figure(figsize=(14, 10))
g = sns.clustermap(heatmap_data, cmap='RdBu_r', center=0, 
                   row_colors=row_colors, row_cluster=False,
                   figsize=(14, 10), dendrogram_ratio=0.1)
g.fig.suptitle('Top Differentially Expressed Genes (Blue=LUAD, Red=LUSC)', y=1.02)
plt.show()

## 9️⃣ Data Quality Summary

In [ ]:
# Summary report
print("=" * 60)
print("DATA QUALITY SUMMARY")
print("=" * 60)
print(f"\n📊 Dataset Shape: {df.shape[0]} samples × {df.shape[1]-1} genes")
print(f"\n📁 Classes:")
for ct in class_counts.index:
    print(f"   - {ct}: {class_counts[ct]} samples ({class_counts[ct]/len(df)*100:.1f}%)")
print(f"\n✅ Class Balance: {class_counts.min()/class_counts.max():.2f} (good, close to 1.0)")
print(f"\n🔍 Missing Values: {missing} ({missing_pct:.2f}%)")
print(f"\n📈 Expression Range: {gene_stats['min'].min():.2f} to {gene_stats['max'].max():.2f}")
print(f"\n🧬 PCA: {n_90} components explain 90% variance")
print("\n" + "=" * 60)
print("DATA IS READY FOR NEXT STEPS!")
print("=" * 60)

---

## 🎯 Next Steps

Based on this EDA, the recommended next steps are:

1. **Gene Filtering**: Remove low-variance genes (keep top ~5,000-10,000)
2. **Gene Mapping**: Map gene symbols to Gene Ontology (GO) terms
3. **Semantic Similarity**: Calculate GO semantic similarity between genes
4. **Feature Engineering**: Create features based on GO similarity
5. **Model Training**: Train classifier (Random Forest, SVM, etc.)

---